# Apple Global Sales Dataset — EDA and Modeling

This notebook is organized in the correct workflow:

1. Import libraries  
2. Load and explore the raw data  
3. Clean the data  
4. Visualize the cleaned data and answer questions  
5. Prepare the data for modeling  
6. Create the model  
7. Evaluate the model  

**Target variable:** `return_status`

## Section 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option("display.max_columns", None)
sns.set_style("whitegrid")

## Section 2: Load and Explore the Raw Data

In [ ]:
df = pd.read_csv("apple_global_sales_dataset.csv")
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

In [ ]:
print("Missing values by column:")
print(df.isnull().sum().sort_values(ascending=False))

print("\nDuplicate rows:", df.duplicated().sum())

### Initial exploration note
At this stage, we only inspect the raw data.  
We do **not** answer analytical questions yet, because missing values and duplicates can affect the results.

## Section 3: Data Cleaning

In [ ]:
df_clean = df.copy()

In [ ]:
# Convert date column to datetime
df_clean["sale_date"] = pd.to_datetime(df_clean["sale_date"], errors="coerce")

In [ ]:
# Handle missing values
df_clean["storage"] = df_clean["storage"].fillna("Unknown")
df_clean["previous_device_os"] = df_clean["previous_device_os"].fillna("Unknown")
df_clean["customer_rating"] = df_clean["customer_rating"].fillna(df_clean["customer_rating"].median())

In [ ]:
# Remove duplicates
df_clean = df_clean.drop_duplicates()

In [ ]:
# Create features from the sale date
df_clean["sale_day"] = df_clean["sale_date"].dt.day
df_clean["sale_dayofweek"] = df_clean["sale_date"].dt.dayofweek

In [ ]:
# Drop columns that are not needed
df_clean = df_clean.drop(columns=["sale_id", "sale_date"])

In [ ]:
print("Shape after cleaning:", df_clean.shape)
print("\nRemaining missing values:")
print(df_clean.isnull().sum().sort_values(ascending=False).head(10))

df_clean.head()

### Cleaning summary
- Converted `sale_date` to datetime  
- Filled missing values in `storage` and `previous_device_os` with `"Unknown"`  
- Filled missing `customer_rating` with the median  
- Removed duplicate rows  
- Dropped `sale_id` and `sale_date` after creating useful date-based features

## Section 4: Visualization and Questions

### 4.1 What is the distribution of return status?

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x="return_status", data=df_clean)
plt.title("Return Status Distribution")
plt.xlabel("Return Status")
plt.ylabel("Count")
plt.show()

print(df_clean["return_status"].value_counts())

**Answer:**  
- Most transactions were **Kept**.  
- Returned and Exchanged orders are much fewer.  
- This means the target variable is somewhat imbalanced.

### 4.2 Which product categories generate the most revenue?

In [ ]:
category_revenue = df_clean.groupby("category")["revenue_usd"].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=category_revenue.index, y=category_revenue.values)
plt.title("Total Revenue by Product Category")
plt.xlabel("Category")
plt.ylabel("Revenue (USD)")
plt.xticks(rotation=45)
plt.show()

category_revenue

**Answer:**  
- The categories with the highest bars generate the most revenue.  
- Premium products usually contribute more to total sales revenue.

### 4.3 Which countries have the highest total revenue?

In [ ]:
top_countries = df_clean.groupby("country")["revenue_usd"].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(10, 5))
sns.barplot(x=top_countries.index, y=top_countries.values)
plt.title("Top 10 Countries by Revenue")
plt.xlabel("Country")
plt.ylabel("Revenue (USD)")
plt.xticks(rotation=45)
plt.show()

top_countries

**Answer:**  
- The top countries contribute the largest share of revenue.  
- These countries represent the strongest markets in the dataset.

### 4.4 Which sales channels sell the most units?

In [ ]:
channel_units = df_clean.groupby("sales_channel")["units_sold"].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=channel_units.index, y=channel_units.values)
plt.title("Units Sold by Sales Channel")
plt.xlabel("Sales Channel")
plt.ylabel("Units Sold")
plt.xticks(rotation=45)
plt.show()

channel_units

**Answer:**  
- The channel with the highest number of units sold performs best in volume.  
- This helps identify the most effective distribution channel.

### 4.5 How are the numerical variables distributed?

In [ ]:
num_cols = [
    "unit_price_usd", "discount_pct", "units_sold",
    "discounted_price_usd", "revenue_usd", "customer_rating"
]

df_clean[num_cols].hist(figsize=(14, 8), bins=25)
plt.suptitle("Distribution of Numerical Features", y=1.02)
plt.show()

**Answer:**  
- These plots help identify spread, skewness, and possible outliers.  
- Revenue and price-related variables may appear right-skewed.

### 4.6 What are the relationships between numerical variables?

In [ ]:
plt.figure(figsize=(8, 6))
corr = df_clean[num_cols].corr()
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

corr

**Answer:**  
- `revenue_usd` is often strongly related to price and units sold.  
- `customer_rating` usually has a weaker relationship with the sales variables.

### 4.7 Which categories have the highest average customer rating?

In [ ]:
rating_by_category = df_clean.groupby("category")["customer_rating"].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=rating_by_category.index, y=rating_by_category.values)
plt.title("Average Customer Rating by Category")
plt.xlabel("Category")
plt.ylabel("Average Rating")
plt.xticks(rotation=45)
plt.show()

rating_by_category

**Answer:**  
- Categories with higher average ratings show better customer satisfaction.  
- Lower-rated categories may need further investigation.

### 4.8 Which categories have more returns or exchanges?

In [ ]:
plt.figure(figsize=(12, 6))
sns.countplot(x="category", hue="return_status", data=df_clean)
plt.title("Return Status by Category")
plt.xlabel("Category")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.show()

**Answer:**  
- Some categories may have more returns or exchanges than others.  
- This can help identify products with more post-purchase issues.

## Section 5: Prepare Data for the Model

In [ ]:
X = df_clean.drop(columns=["return_status"])
y = df_clean["return_status"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nTarget distribution:")
print(y.value_counts())

In [ ]:
num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object"]).columns.tolist()

print("Numerical features:")
print(num_features)

print("\nCategorical features:")
print(cat_features)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set shape:", X_train.shape)
print("Testing set shape:", X_test.shape)

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_features),
    ("cat", categorical_transformer, cat_features)
])

## Section 6: Create the Model

In [ ]:
model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        class_weight="balanced"
    ))
])

model.fit(X_train, y_train)

## Section 7: Make Predictions

In [ ]:
y_pred = model.predict(X_test)

## Section 8: Evaluate the Model

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.named_steps["classifier"].classes_)

plt.figure(figsize=(6, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=model.named_steps["classifier"].classes_,
    yticklabels=model.named_steps["classifier"].classes_
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()

## Section 9: Feature Importance

In [ ]:
ohe = model.named_steps["preprocessor"].named_transformers_["cat"].named_steps["onehot"]
encoded_cat_names = ohe.get_feature_names_out(cat_features)

all_feature_names = num_features + list(encoded_cat_names)
importances = model.named_steps["classifier"].feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": all_feature_names,
    "importance": importances
}).sort_values(by="importance", ascending=False)

feature_importance_df.head(15)

In [ ]:
top15 = feature_importance_df.head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x="importance", y="feature", data=top15)
plt.title("Top 15 Important Features")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()

## Section 10: Final Conclusions

- The raw data was explored first to understand its structure, missing values, and duplicates.  
- Data cleaning was completed before answering analytical questions to improve reliability.  
- Most transactions were **Kept**, while returns and exchanges were less common.  
- Revenue varied across categories, countries, and sales channels.  
- A Random Forest model was built to predict `return_status`.  
- Feature importance helped identify which variables contributed most to the model.